# Day 5: Data Cleaning and Preprocessing

Before any model touches data, the data has to be cleaned and shaped. This notebook works through the preparation stage end to end.

- What cleaning and preprocessing involve
- Missing values, duplicates and inconsistent text
- Column selection, type casting and transformations
- Categorical encoding
- A full cleaning pass on a real rainfall dataset
- A messy student records dataset made model ready

Garbage data in gives garbage results out, so investment in this stage pays for itself later.

In [1]:
import numpy as np
import pandas as pd

## What Data Cleaning Covers

- Fixing incorrect or impossible values
- Handling missing entries
- Removing duplicate rows
- Standardizing text and formats

Preprocessing matters because ML models demand consistent numeric input; messy columns break training or silently skew it.

A tiny example of the kind of table that causes trouble:

| Name | Age | City |
|------|-----|------|
| Ravi | 25 | Chennai |
| Kiran | NaN | chennai |
| Ravi | 25 | Chennai |

Three problems in six cells: a missing age, two spellings of one city, and a repeated row.

## First Messy Example

In [2]:
data = {
    "Name": ["Ravi", "Kiran", "Ravi", "Anita", None, "Kiran", "Kiran", "Kiran", "Ravi"],
    "Age": [25, np.nan, 25, 30, 22, 13, 17, 18, 19],
    "City": ["Chennai", "chennai", "Chennai", "Delhi", "Delhi", "Delhi", "Delhi", "Delhi", "Delhi"],
}

df = pd.DataFrame(data)
df

,Name,Age,City
0,Ravi,25.0,Chennai
1,Kiran,NaN,chennai
2,Ravi,25.0,Chennai
3,Anita,30.0,Delhi
4,None,22.0,Delhi
5,Kiran,13.0,Delhi
6,Kiran,17.0,Delhi
7,Kiran,18.0,Delhi
8,Ravi,19.0,Delhi


## Common Data Problems

1. Missing values, shown as NaN or None
2. Duplicate records
3. Wrong data types, for example numbers stored as text
4. Inconsistent capitalization, like Chennai and chennai

Each of these gets identified first and fixed second.

## Initial Inspection

### First Rows

In [3]:
df.head()

,Name,Age,City
0,Ravi,25.0,Chennai
1,Kiran,NaN,chennai
2,Ravi,25.0,Chennai
3,Anita,30.0,Delhi
4,None,22.0,Delhi


### Structure

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    8 non-null      object 
 1   Age     8 non-null      float64
 2   City    9 non-null      object 
dtypes: float64(1), object(2)
memory usage: 348.0+ bytes


### Summary Statistics

In [5]:
df.describe()

,Age
count,8.00000
mean,21.12500
std,5.43632
min,13.00000
25%,17.75000
50%,20.50000
75%,25.00000
max,30.00000


## Missing Values

Missing entries appear as NaN or None. The routine is: detect them, count them, then either drop the rows or fill the gaps with a sensible value.

In [6]:
df.isnull()

,Name,Age,City
0,False,False,False
1,False,True,False
2,False,False,False
3,False,False,False
4,True,False,False
5,False,False,False
6,False,False,False
7,False,False,False
8,False,False,False


In [7]:
df.isnull().sum()

,0
Name,1
Age,1
City,0


### Dropping Rows with Missing Values

In [8]:
df_drop = df.dropna()
df_drop

,Name,Age,City
0,Ravi,25.0,Chennai
2,Ravi,25.0,Chennai
3,Anita,30.0,Delhi
5,Kiran,13.0,Delhi
6,Kiran,17.0,Delhi
7,Kiran,18.0,Delhi
8,Ravi,19.0,Delhi


### Filling Missing Values

The usual choices:

- Numerical columns: mean for balanced data, median when the column is skewed
- Categorical columns: mode, the most frequent value

Mean works for the Age column here, and the Name column takes its mode.

In [9]:
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Name"] = df["Name"].fillna(df["Name"].mode()[0])

df

,Name,Age,City
0,Ravi,25.000,Chennai
1,Kiran,21.125,chennai
2,Ravi,25.000,Chennai
3,Anita,30.000,Delhi
4,Kiran,22.000,Delhi
5,Kiran,13.000,Delhi
6,Kiran,17.000,Delhi
7,Kiran,18.000,Delhi
8,Ravi,19.000,Delhi


## Working with Multiple Columns at Once

A second table shows how one operation can sweep across several columns. Selecting a group of columns keeps the code short and applies every change consistently.

In [10]:
df = pd.DataFrame({
    "Age": [20, np.nan, 25, 30, 20],
    "Salary": [20000, 25000, np.nan, 40000, 20000],
    "City": ["Nashik", "Delhi", "Pune", "Akola", "Mumbai"],
})

df

,Age,Salary,City
0,20.0,20000.0,Nashik
1,NaN,25000.0,Delhi
2,25.0,NaN,Pune
3,30.0,40000.0,Akola
4,20.0,20000.0,Mumbai


### Filling Several Columns Together

In [11]:
df[["Age", "Salary"]] = df[["Age", "Salary"]].fillna(df[["Age", "Salary"]].mean())

df

,Age,Salary,City
0,20.00,20000.0,Nashik
1,23.75,25000.0,Delhi
2,25.00,26250.0,Pune
3,30.00,40000.0,Akola
4,20.00,20000.0,Mumbai


### Type Casting

In [12]:
df[["Age", "Salary"]] = df[["Age", "Salary"]].astype(int)

df.dtypes

,0
Age,int64
Salary,int64
City,object


### Applying a Calculation to Every Column

In [13]:
# every number goes up by 10 percent
df[["Age", "Salary"]] = df[["Age", "Salary"]].apply(lambda x: x * 1.1)

df

,Age,Salary,City
0,22.0,22000.0,Nashik
1,25.3,27500.0,Delhi
2,27.5,28875.0,Pune
3,33.0,44000.0,Akola
4,22.0,22000.0,Mumbai


### Replacing Specific Values

In [14]:
df[["City"]] = df[["City"]].replace("Mumbai", "Pune")

df

,Age,Salary,City
0,22.0,22000.0,Nashik
1,25.3,27500.0,Delhi
2,27.5,28875.0,Pune
3,33.0,44000.0,Akola
4,22.0,22000.0,Pune


## Encoding Categorical Columns

Models work with numbers only, so text categories have to become numeric.

- Nominal data has no order, like City or Color
- Ordinal data has a ranking, like Low, Medium, High

Two standard methods:

- One-hot encoding: one boolean column per category
- Label encoding: one integer per category

### One-Hot Encoding

In [15]:
df_encoded = pd.get_dummies(df, columns=["City"])
df_encoded

,Age,Salary,City_Akola,City_Delhi,City_Nashik,City_Pune
0,22.0,22000.0,False,False,True,False
1,25.3,27500.0,False,True,False,False
2,27.5,28875.0,False,False,False,True
3,33.0,44000.0,True,False,False,False
4,22.0,22000.0,False,False,False,True


### Label Encoding

Label encoding assigns one number per category, for example Chennai to 0 and Delhi to 1. It is compact but invents an order between unordered categories, so it suits ordinal columns better than nominal ones.

In [16]:
rank_manual = df["City"].map({"Nashik": 0, "Delhi": 1, "Pune": 2})

pd.DataFrame({"City": df["City"], "Code": rank_manual})

,City,Code
0,Nashik,0.0
1,Delhi,1.0
2,Pune,2.0
3,Akola,NaN
4,Pune,2.0


## Duplicate Records

Duplicates bias a model by over-representing certain rows. They are counted, removed with drop_duplicates, and inconsistent capitalization is unified at the same time.

In [17]:
df_no_dup = df.drop_duplicates()
df_no_dup

,Age,Salary,City
0,22.0,22000.0,Nashik
1,25.3,27500.0,Delhi
2,27.5,28875.0,Pune
3,33.0,44000.0,Akola
4,22.0,22000.0,Pune


In [18]:
# unify the case of every city name
df["City"] = df["City"].str.upper()
df.dropna()

df

,Age,Salary,City
0,22.0,22000.0,NASHIK
1,25.3,27500.0,DELHI
2,27.5,28875.0,PUNE
3,33.0,44000.0,AKOLA
4,22.0,22000.0,PUNE


## Real Dataset: Cleaning the Rainfall Data

The rainfall CSV holds daily weather records with these columns: date, rainfall, temperature, humidity, wind_speed and weather_condition. Known issues: unparsed dates, missing values, condition text in mixed case, occasional invalid rows and wrong types.

The cleaning order:

1. Load the dataset
2. Convert the date column
3. Handle missing values
4. Fix text inconsistencies
5. Remove duplicates
6. Detect and handle outliers
7. Save the final clean dataset

### Downloading the Dataset

The file is fetched from Kaggle through kagglehub into a local cache, so no manual upload step is needed.

In [19]:
import os
import kagglehub

path = kagglehub.dataset_download("sujithmandala/simple-rainfall-classification-dataset")
print("Dataset folder:", path)

csv_file = None
for root, dirs, files in os.walk(path):
    for file_name in files:
        if file_name.endswith(".csv"):
            csv_file = os.path.join(root, file_name)

print("CSV found at:", csv_file)

100%|██████████| 804/804 [00:00<00:00, 1.20MB/s]

Extracting files...
Dataset folder: /root/.cache/kagglehub/datasets/sujithmandala/simple-rainfall-classification-dataset/versions/1
CSV found at: /root/.cache/kagglehub/datasets/sujithmandala/simple-rainfall-classification-dataset/versions/1/rainfall.csv


In [20]:
df = pd.read_csv(csv_file)
df.head()

,date,rainfall,temperature,humidity,wind_speed,weather_condition
0,2022-01-01,12.5,15.2,78.0,8.5,Rainy
1,2022-01-02,8.2,17.8,65.0,5.2,Rainy
2,2022-01-03,0.0,20.1,52.0,3.1,Sunny
3,2022-01-04,3.7,18.6,71.0,6.7,Rainy
4,2022-01-05,21.1,14.8,82.0,9.3,Rainy


### Inspection

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               54 non-null     object 
 1   rainfall           53 non-null     float64
 2   temperature        53 non-null     float64
 3   humidity           53 non-null     float64
 4   wind_speed         53 non-null     float64
 5   weather_condition  53 non-null     object 
dtypes: float64(4), object(2)
memory usage: 2.7+ KB


In [22]:
df.isnull().sum()

,0
date,0
rainfall,1
temperature,1
humidity,1
wind_speed,1
weather_condition,1


A few rows carry a short placeholder like Co instead of a real condition value; those behave like garbage text and get standardized along with the casing fix below.

### Converting the Date Column

The date arrives as plain text. Converting it to a proper datetime type makes sorting and time based analysis possible.

In [23]:
df["date"] = df["date"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               53 non-null     datetime64[ns]
 1   rainfall           53 non-null     float64       
 2   temperature        53 non-null     float64       
 3   humidity           53 non-null     float64       
 4   wind_speed         53 non-null     float64       
 5   weather_condition  53 non-null     object        
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 2.7+ KB


### Filling Missing Values

Numeric columns take the column mean, the categorical condition column takes its mode.

In [24]:
num_cols = ["rainfall", "temperature", "humidity", "wind_speed"]

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
df["weather_condition"] = df["weather_condition"].fillna(df["weather_condition"].mode()[0])

df.isnull().sum()

,0
date,1
rainfall,0
temperature,0
humidity,0
wind_speed,0
weather_condition,0


### Unifying Text Case

In [25]:
df["weather_condition"] = df["weather_condition"].str.lower()

df["weather_condition"].unique()

array(['rainy', 'sunny'], dtype=object)

In [26]:
df.head()

,date,rainfall,temperature,humidity,wind_speed,weather_condition
0,2022-01-01,12.5,15.2,78.0,8.5,rainy
1,2022-01-02,8.2,17.8,65.0,5.2,rainy
2,2022-01-03,0.0,20.1,52.0,3.1,sunny
3,2022-01-04,3.7,18.6,71.0,6.7,rainy
4,2022-01-05,21.1,14.8,82.0,9.3,rainy


### Removing Duplicates

In [27]:
print("Duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Rows now:", len(df))

Duplicate rows: 0
Rows now: 54


### Outlier Check

The IQR rule flags anything sitting more than 1.5 spread lengths outside the middle fifty percent of a column.

In [28]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    spread = Q3 - Q1
    lo, hi = Q1 - 1.5 * spread, Q3 + 1.5 * spread
    print(col, "outliers:", ((df[col] < lo) | (df[col] > hi)).sum())

rainfall outliers: 0
temperature outliers: 0
humidity outliers: 0
wind_speed outliers: 0


### Saving the Clean Dataset

In [29]:
df.to_csv("cleaned_rainfall.csv", index=False)
print("Saved cleaned_rainfall.csv,", len(df), "rows")

Saved cleaned_rainfall.csv, 54 rows


In [30]:
try:
    df.to_excel("cleaned_rainfall.xlsx", index=False)
    print("Saved cleaned_rainfall.xlsx")
except ImportError:
    print("openpyxl is not installed here, the Excel export was skipped")

Saved cleaned_rainfall.xlsx


## Real Dataset: Cleaning Student Records

A second table, this time deliberately broken: absent names, a missing age, gender written four different ways, marks outside the legal range and one entirely bad row.

In [31]:
df = pd.DataFrame({
    "name": ["Amit", "Sara", "John", "Amit", None],
    "age": [20, 21, np.nan, 20, 19],
    "gender": ["Male", "female", "Male", "Male", "FEMALE"],
    "marks": [85, 90, 105, 85, -5],
    "attendance": [90, 85, 80, 90, None],
    "city": ["Mumbai", "Delhi", "mumbai", "Mumbai", "Pune"],
})

df

,name,age,gender,marks,attendance,city
0,Amit,20.0,Male,85,90.0,Mumbai
1,Sara,21.0,female,90,85.0,Delhi
2,John,NaN,Male,105,80.0,mumbai
3,Amit,20.0,Male,85,90.0,Mumbai
4,None,19.0,FEMALE,-5,NaN,Pune


### Inspection

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        4 non-null      object 
 1   age         4 non-null      float64
 2   gender      5 non-null      object 
 3   marks       5 non-null      int64  
 4   attendance  4 non-null      float64
 5   city        5 non-null      object 
dtypes: float64(2), int64(1), object(3)
memory usage: 372.0+ bytes


In [33]:
df.isnull().sum()

,0
name,1
age,1
gender,0
marks,0
attendance,1
city,0


In [34]:
df.describe()

,age,marks,attendance
count,4.000000,5.000000,4.000000
mean,20.000000,72.000000,86.250000
std,0.816497,43.817805,4.787136
min,19.000000,-5.000000,80.000000
25%,19.750000,85.000000,83.750000
50%,20.000000,85.000000,87.500000
75%,20.250000,90.000000,90.000000
max,21.000000,105.000000,90.000000


Marks show a maximum of 105 and a minimum of -5, both impossible, and two columns have gaps.

### Filling the Gaps

Numeric columns take the mean and the name column takes a fixed label, since no single real name would be fair.

In [35]:
df["age"] = df["age"].fillna(df["age"].mean())
df["attendance"] = df["attendance"].fillna(df["attendance"].mean())
df["name"] = df["name"].fillna("Unknown")

df

,name,age,gender,marks,attendance,city
0,Amit,20.0,Male,85,90.00,Mumbai
1,Sara,21.0,female,90,85.00,Delhi
2,John,20.0,Male,105,80.00,mumbai
3,Amit,20.0,Male,85,90.00,Mumbai
4,Unknown,19.0,FEMALE,-5,86.25,Pune


### Unifying Text

In [36]:
df["gender"] = df["gender"].str.lower()
df["city"] = df["city"].str.lower()

df

,name,age,gender,marks,attendance,city
0,Amit,20.0,male,85,90.00,mumbai
1,Sara,21.0,female,90,85.00,delhi
2,John,20.0,male,105,80.00,mumbai
3,Amit,20.0,male,85,90.00,mumbai
4,Unknown,19.0,female,-5,86.25,pune


### Filtering Invalid Marks

Rows whose marks fall outside 0 to 100 are dropped by boolean filtering.

In [37]:
df = df[(df["marks"] >= 0) & (df["marks"] <= 100)]
df = df.drop_duplicates()

df

,name,age,gender,marks,attendance,city
0,Amit,20.0,male,85,90.0,mumbai
1,Sara,21.0,female,90,85.0,delhi


## Practice: Basic Questions

### Q1. Load the cleaned CSV and show the first and last five rows

In [38]:
check = pd.read_csv("cleaned_rainfall.csv")
print(check.head())
print(check.tail())

         date  rainfall  temperature  humidity  wind_speed weather_condition
0  2022-01-01      12.5         15.2      78.0         8.5             rainy
1  2022-01-02       8.2         17.8      65.0         5.2             rainy
2  2022-01-03       0.0         20.1      52.0         3.1             sunny
3  2022-01-04       3.7         18.6      71.0         6.7             rainy
4  2022-01-05      21.1         14.8      82.0         9.3             rainy
          date   rainfall  temperature   humidity  wind_speed  \
49  2022-02-19  17.300000    15.900000  83.000000    8.600000   
50  2022-02-20   9.600000    18.600000  71.000000    6.700000   
51  2022-02-21   0.000000    21.300000  55.000000    5.100000   
52  2022-02-22   7.400000    16.800000  77.000000    8.900000   
53         NaN   9.049057    17.950943  69.245283    6.403774   

   weather_condition  
49             rainy  
50             rainy  
51             sunny  
52             rainy  
53             rainy  


### Q2. Show the total rows and columns plus the type of every column

In [39]:
print("Rows and columns:", check.shape)
print(check.dtypes)

Rows and columns: (54, 6)
date                  object
rainfall             float64
temperature          float64
humidity             float64
wind_speed           float64
weather_condition     object
dtype: object


### Q3. Report which columns contain missing values and how many

In [40]:
missing = check.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values remain")

date    1
dtype: int64


### Q4. Fill missing numeric values with the mean and categorical with the mode

In [41]:
# demonstration on a fresh messy table
practice = pd.DataFrame({
    "temp": [30, np.nan, 28, 31],
    "sky": ["clear", None, "clear", "rain"],
})

practice["temp"] = practice["temp"].fillna(practice["temp"].mean())
practice["sky"] = practice["sky"].fillna(practice["sky"].mode()[0])
practice

,temp,sky
0,30.000000,clear
1,29.666667,clear
2,28.000000,clear
3,31.000000,rain


### Q5. Remove duplicate rows

In [42]:
messy = pd.DataFrame({"a": [1, 2, 2, 3], "b": ["x", "y", "y", "z"]})
messy.drop_duplicates()

,a,b
0,1,x
1,2,y
3,3,z


### Q6. Convert a date column into datetime format

In [43]:
sample = pd.DataFrame({"date": ["2024-04-01 ", "2024-04-02"]})
sample["date"] = pd.to_datetime(sample["date"].str.strip())
sample.dtypes

,0
date,datetime64[ns]


### Q7. Rename all columns to lowercase

In [44]:
wide = pd.DataFrame({"Marks ": [1], "Name ": ["A"]})
wide.columns = wide.columns.str.strip().str.lower()
wide.columns

Index(['marks', 'name'], dtype='object')

### Q8. Convert float columns to int where the data allows it

In [45]:
casts = pd.DataFrame({"age": [20.0, 21.0, 22.0], "pct": [10.5, 20.5, 30.5]})
casts["age"] = casts["age"].astype(int)
casts.dtypes

,0
age,int64
pct,float64


## Practice: Intermediate Questions

### Q1. Select multiple columns and fill their missing values in one step

In [46]:
group = pd.DataFrame({
    "marks": [70, np.nan, 80],
    "attendance": [np.nan, 60, 90],
})

group[["marks", "attendance"]] = group[["marks", "attendance"]].fillna(
    group[["marks", "attendance"]].mean()
)
group

,marks,attendance
0,70.0,75.0
1,75.0,60.0
2,80.0,90.0


### Q2. Raise every salary value by 10 percent

In [47]:
pay = pd.DataFrame({"salary": [30000, 45000, 52000]})
pay["salary"] = pay["salary"].apply(lambda v: v * 1.1)
pay

,salary
0,33000.0
1,49500.0
2,57200.0


### Q3. Standardize mixed-case city spellings into one format

In [48]:
cities = pd.DataFrame({"city": ["mumbai", "MUMBAI", "Mumbai", "mumbai "]})
cities["city"] = cities["city"].str.strip().str.capitalize()
cities

,city
0,Mumbai
1,Mumbai
2,Mumbai
3,Mumbai


### Q4. Build a Pass or Fail column from the marks

In [49]:
result_df = df.copy()
result_df["result"] = np.where(result_df["marks"] >= 50, "Pass", "Fail")
result_df

,name,age,gender,marks,attendance,city,result
0,Amit,20.0,male,85,90.0,mumbai,Pass
1,Sara,21.0,female,90,85.0,delhi,Pass


### Q5. Filter rows, students with marks above 80 and rainfall above 10

In [50]:
print("Students above 80:")
print(result_df[result_df["marks"] > 80])

print("Rainfall above 10:")
print(check[check["rainfall"] > 10].head())

Students above 80:
   name   age  gender  marks  attendance    city result
0  Amit  20.0    male     85        90.0  mumbai   Pass
1  Sara  21.0  female     90        85.0   delhi   Pass
Rainfall above 10:
         date  rainfall  temperature  humidity  wind_speed weather_condition
0  2022-01-01      12.5         15.2      78.0         8.5             rainy
4  2022-01-05      21.1         14.8      82.0         9.3             rainy
5  2022-01-06      15.3         16.5      75.0         7.8             rainy
8  2022-01-09      11.2         17.3      73.0         6.1             rainy
9  2022-01-10      18.6         15.8      79.0         8.9             rainy


### Q6. Drop a column that is not needed

In [51]:
slim = check.drop(columns=["wind_speed"])
slim.columns

Index(['date', 'rainfall', 'temperature', 'humidity', 'weather_condition'], dtype='object')

### Q7. One-hot encode a categorical column

In [52]:
pd.get_dummies(check, columns=["weather_condition"]).head()2

,date,rainfall,temperature,humidity,wind_speed,weather_condition_rainy,weather_condition_sunny
0,2022-01-01,12.5,15.2,78.0,8.5,True,False
1,2022-01-02,8.2,17.8,65.0,5.2,True,False
2,2022-01-03,0.0,20.1,52.0,3.1,False,True
3,2022-01-04,3.7,18.6,71.0,6.7,True,False
4,2022-01-05,21.1,14.8,82.0,9.3,True,False


### Q8. Sort the dataset by a column of your choice

In [53]:
check.sort_values(by="temperature", ascending=False).head()

,date,rainfall,temperature,humidity,wind_speed,weather_condition
41,2022-02-11,0.0,23.4,44.0,3.1,sunny
26,2022-01-27,0.0,22.9,47.0,2.5,sunny
46,2022-02-16,0.0,22.7,51.0,4.5,sunny
12,2022-01-13,0.0,22.4,45.0,2.1,sunny
36,2022-02-06,0.0,22.1,49.0,2.7,sunny
